# Lymphoid (T/NK/B) Fine Anotation

Clusters the lymphoid compartment on the scVI latent space, assigns cell states from
cluster markers (logistic regression, TF-IDF) and curated panels, and summarises state
composition per sample and RECIST treatment stage.

In [ ]:
# !pip install --quiet scanpy leidenalg igraph scikit-learn
# %cd /content/drive/MyDrive/Colab Notebooks/data/h5ad/lymphoid

In [ ]:
import os

import anndata
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
from sklearn.feature_extraction.text import TfidfTransformer

## Config

In [ ]:
input_path = "sub_Lymphoid_raw_scvi_combine.h5ad"

output_path = "sub_Lymphoid_raw_scvi_combine_annotated.h5ad"

annotation_csv = "tables/lymphoid_cell_state_annotation.csv"
figures_output_dir = "./figures"
tables_output_dir = "./tables"

counts_layer = "raw_counts"
sample_col = "ID_2"
recist_col = "recist_v3"

# --- clustering ---
use_rep = "X_scVI_n_latent_hvg_10"
n_neighbors = 40
leiden_resolution = 0.8
seed = 0

liver_biopsy_ids = ["R008", "R011"]  # matched on obs["ID"]


count_axis_break = 250

# --- marker panels ---
marker_genes = {
    "CD4": ["CD3D", "CD3E", "CXCR4", "LTB", "CD4", "TRBC2"],
    "naive": ["CCR7", "SELL", "LEF1", "TCF7"],
    "CD4 effector memory": ["IL7R", "FAS"],
    "CD8": ["CD8A", "CD8B"],
    "T cell dividing": ["TOP2A", "EZH2", "MKI67", "CENPP"],
    "Treg": ["IKZF2", "IL2RA", "ICOS", "TOX2", "FOXP3", "TNFRSF9"],
    "cytotoxic/effector": ["PDCD1", "CD226", "BACH2", "NECTIN3", "KLRG1", "GZMA", "GZMB", "PRF1", "NKG7",
                           "FCGR3A", "CX3CR1", "CD28"],
    "dysfunctional/exhaustive": ["LAYN", "CTLA4", "HAVCR2", "PDCD1", "LAG3", "TIGIT"],
    "Th17": ["CCR6", "CCR4"],
    "Trm": ["ITGA1", "ITGAE", "CXCR6", "CD69"],
    "Bystander": ["ENTPD1"],
    "activating": ["GNLY", "GZMK", "ZNF683", "IFNG", "IL2", "TNF"],
    "NKcell": ["ZEB2", "KLRD1", "NCAM1", "KLRB1"],
    "TF": ["RORC", "EOMES", "HOPX", "TBX21", "ZEB2", "ZNF683", "HIF1A", "ID2", "TOX"],
    "GEP": ["HLA-DQA1", "PSMB10", "HLA-DRB1", "HLA-E", "CD276", "CD38", "CXCL9", "IDO1", "STAT1",
            "PDCD1LG2", "CD274", "CMKLR1", "CXCR6", "CCL5", "CD27"],
    "Bcell": ["CD19", "CD79A", "MS4A1"],
    "Plasma_cell": ["CD27", "SDC1", "CD38", "IRF4"],
    "MAIT/Tc17": ["KLRB1", "ZBTB16", "TRAV1-2", "SLC4A10", "IL17A", "IL17F"],
}

marker_genes_ilc3 = {"ILC3": ["IL23R", "IL1R1", "KIT", "TNFSF13B", "IL17A", "IL22", "CSF2", "AHR", "TOX",
                              "ZBTB16", "HLA-DRA"]}

markers_1 = {                   # saved as dotplot_markers_1*.pdf
    "Plasma_cell": ["IGKC", "SDC1", "CD38"], "Bcell": ["CD19", "MS4A1"], "MemB": ["CD27", "CR2", "FCER2"],
    "Proliferating": ["TOP2A", "MKI67", "CENPP"], "NKcell": ["KLRD1", "NCAM1"],
    "NKcell-cyt": ["NKG7", "GZMB", "GNLY", "PRF1"], "CD4/CD8": ["CCR7", "SELL", "IL7R", "CD28", "LEF1", "TCF7"],
    "Tem": ["GZMK", "GZMA", "KLRG1"], "Treg": ["IL2RA", "FOXP3"], "Bystander": ["ENTPD1"],
    "Trm": ["CD69", "ITGA1", "ITGAE", "CXCR6"],
}
markers_2 = {                   # saved as dotplot_markers_2*.pdf
    "cytotoxic/effector1": ["KLRG1", "GZMA", "GZMB", "PRF1", "NKG7", "FCGR3A", "CX3CR1", "CCL5", "GNLY", "HAVCR2"],
    "cytotoxic/effector2": ["LAG3", "GZMK"], "Bystander": ["ENTPD1"],
    "inhibitory": ["LAYN", "TIGIT", "CTLA4", "PDCD1"], "co-stim1": ["CD28", "CD40LG", "CD226"],
    "co-stim2": ["ICOS", "CD27", "TNFRSF4", "TNFRSF9", "TNFRSF18"],
}
gene_panels = {                 # UMAP panels; obs columns in label_keys are plotted as-is
    "tls_nk": ["SFTPB", "MS4A1", "BCL6", "AICDA", "LAMP3", "CCR6", "GNLY", "NKG7", "PRF1", "NCAM1", "IL7R",
               "TIGIT", "LAG3", "TOX", "cell_state"],
    "lineage": ["ENTPD1", "CD4", "CD8A", "CD8B", "CD19", "CD79A", "GZMA", "GZMK", "PRF1", "IL7R", "cell_state"],
}

label_keys = {"leiden_scvi", "cell_state", "cell_state_v2", "celltype_predictions"}

min_detection_pct = 1.0

for d in (figures_output_dir, tables_output_dir):
    os.makedirs(d, exist_ok=True)
sc.settings.figdir = figures_output_dir
sc.settings.verbosity = 1
sc.set_figure_params(dpi=80, figsize=(4, 4))

## Helpers

In [ ]:
def check_object(ad, stage):
    print(f"[{stage}] {ad.n_obs} cells x {ad.n_vars} genes | samples: {ad.obs[sample_col].nunique()}")
    for col in ("cell_state", "cell_state_v2"):
        if col in ad.obs:
            n_na = ad.obs[col].isna().sum()
            print(f"  {col}: {ad.obs[col].nunique()} labels, {n_na} unlabelled")
            assert n_na == 0, f"{col} has unlabelled cells"

def present(panel, ad):
    if isinstance(panel, dict):
        return {k: [g for g in dict.fromkeys(v) if g in ad.var_names] for k, v in panel.items()
                if any(g in ad.var_names for g in v)}
    return [k for k in dict.fromkeys(panel) if k in ad.var_names or k in ad.obs.columns]

def gene_panel_sanity(ad):
    X = ad.layers[counts_layer]
    panels = {**{f"umap:{k}": v for k, v in gene_panels.items()},
              **{f"dotplot:{name}:{k}": v for name, p in (("markers", marker_genes), ("ilc3", marker_genes_ilc3),
                                                         ("markers_1", markers_1), ("markers_2", markers_2))
                 for k, v in p.items()}}
    rows = []
    for panel, genes in panels.items():
        counts = pd.Series(genes).value_counts()
        for gene in dict.fromkeys(genes):
            if gene in label_keys or gene in ad.obs.columns:
                continue
            in_data = gene in ad.var_names
            pct = float((X[:, ad.var_names.get_loc(gene)] > 0).mean() * 100) if in_data else np.nan
            rows.append({"panel": panel, "gene": gene, "in_data": in_data, "duplicated_in_list": counts[gene] > 1,
                         "pct_cells_detected": pct, "low_detection": in_data and pct < min_detection_pct})
    return pd.DataFrame(rows)

def tfidf_ranking(ad, groupby):
    binary = (ad.X > 0).astype(np.float32) if sp.issparse(ad.X) else sp.csr_matrix(ad.X > 0, dtype=np.float32)
    ad_bin = anndata.AnnData(X=TfidfTransformer().fit_transform(binary.T).T.tocsr(), obs=ad.obs.copy(), var=ad.var.copy())
    sc.tl.rank_genes_groups(ad_bin, groupby=groupby, use_raw=False, method="wilcoxon")
    return ad_bin

def rank_markers(ad, groupby, n_genes=20):
    """Logistic-regression and TF-IDF markers per group, unfiltered and specificity-filtered."""
    if isinstance(ad.obs[groupby].dtype, pd.CategoricalDtype):
        ad.obs[groupby] = ad.obs[groupby].cat.remove_unused_categories()
    sc.tl.rank_genes_groups(ad, groupby, method="logreg", key_added="logreg", max_iter=1000)
    ad_tfidf = tfidf_ranking(ad, groupby)
    sc.tl.filter_rank_genes_groups(ad, min_in_group_fraction=0.2, max_out_group_fraction=0.2,
                                   key="logreg", key_added="logreg_filtered")
    sc.tl.filter_rank_genes_groups(ad_tfidf, min_in_group_fraction=0.2, max_out_group_fraction=0.2,
                                   key_added="tfidf_filtered")
    for obj, key in ((ad, "logreg"), (ad, "logreg_filtered"), (ad_tfidf, "rank_genes_groups"), (ad_tfidf, "tfidf_filtered")):
        if pd.DataFrame(obj.uns[key]["names"]).isna().all(axis=None):
            print(f"{key}: no genes pass the specificity filter; skipped")
            continue
        sc.pl.rank_genes_groups_dotplot(obj, groupby=groupby, n_genes=n_genes, key=key,
                                        standard_scale="var", dendrogram=False, title=key)

def italic_dotplot(ad, panel, groupby, figsize, save_path=None, **kwargs):
    axes_dict = sc.pl.dotplot(ad, present(panel, ad), groupby=groupby, use_raw=False, standard_scale="var",
                              cmap="Blues", figsize=figsize, show=False, **kwargs)
    for ax in axes_dict.values():
        for tick in ax.get_xticklabels():
            tick.set_style("italic")
    if save_path:
        plt.savefig(save_path, bbox_inches="tight")
    plt.show()

def proportion_barplot(ad, which_var, by, colors, order=None, category_order=None, masked=None,
                       horizontal=False, figsize=(11, 2.5), save_name=None):
    data = pd.crosstab(ad.obs[by], ad.obs[which_var], normalize="index") * 100
    if category_order:
        data = data[[c for c in category_order if c in data.columns]]
    if masked:
        data.loc[data.index.isin(masked), :] = 0
    if order:
        data = data.reindex([o for o in order if o in data.index])
    ax = (data.plot.barh if horizontal else data.plot.bar)(stacked=True, edgecolor="none", figsize=figsize,
                                                           fontsize=12, grid=False, width=0.85,
                                                           color=[colors[c] for c in data.columns])
    (ax.set_xlim if horizontal else ax.set_ylim)(0, 100)
    ax.set_title(f"{which_var} %") if horizontal else ax.set_ylabel(f"{which_var} %")
    ax.legend(bbox_to_anchor=(1, 1), loc="upper left", frameon=False)
    if save_name:
        ax.figure.savefig(f"{figures_output_dir}/{save_name}", bbox_inches="tight", dpi=300)
    plt.show()

## 1. Load and gene sanity check

In [ ]:
adata = sc.read(input_path)
sample = adata.X[:1000].toarray() if sp.issparse(adata.X) else np.asarray(adata.X[:1000])
assert np.allclose(sample, np.round(sample)) and sample.min() >= 0, "expected raw counts"
adata.layers[counts_layer] = adata.X.copy()
check_object(adata, "loaded")

gene_check = gene_panel_sanity(adata)
gene_check.to_csv(f"{tables_output_dir}/lymphoid_gene_panel_sanity_check.csv", index=False)
print("missing:", sorted(gene_check.loc[~gene_check.in_data, "gene"].unique()))
print("duplicated within a panel:", gene_check.loc[gene_check.duplicated_in_list, ["panel", "gene"]].values.tolist())
print(f"detected in < {min_detection_pct}% of cells:", sorted(gene_check.loc[gene_check.low_detection, "gene"].unique()))

## 2. Normalise and cluster

In [ ]:
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

sc.pp.neighbors(adata, n_neighbors=n_neighbors, use_rep=use_rep, random_state=seed)
sc.tl.umap(adata, random_state=seed)

sc.tl.leiden(adata, key_added="leiden_scvi", resolution=leiden_resolution, random_state=seed, flavor="leidenalg")

qc_umap_keys = ["leiden_scvi", "phase", "n_genes", "percent_mito", "recist_v2", "recist_v3", "ID_2",
                "GEP_score", "celltype_predictions"]

sc.pl.umap(adata, color=[k for k in qc_umap_keys if k in adata.obs], ncols=3, wspace=0.3, size=60, color_map="OrRd")

## 3. Cluster markers and reference labels

In [ ]:
rank_markers(adata, "leiden_scvi")
sc.pl.dotplot(adata, present(marker_genes, adata), groupby="leiden_scvi", use_raw=False,
              standard_scale="var", cmap="Blues", figsize=(24, 5), dendrogram=True)
sc.pl.dotplot(adata, present(marker_genes_ilc3, adata), groupby="leiden_scvi", use_raw=False,
              standard_scale="var", cmap="Blues", figsize=(8, 5), dendrogram=True)

reference_groups = [            # celltype_predictions groups highlighted on the UMAP
    "T cell CD4", "T cell CD8 naive", "T cell regulatory", "T cell CD8 terminally exhausted",
    "T cell CD8 effector memory", "T cell CD8 activated", "T cell NK-like", "B cell", "B cell dividing",
    "Plasma cell", "NK cell",
]

if "celltype_predictions" in adata.obs:
    for group in reference_groups:
        if group in set(adata.obs["celltype_predictions"]):
            sc.pl.umap(adata, color="celltype_predictions", groups=[group], size=105, title=group)
            
sc.pl.umap(adata, color=sample_col, groups=[s for s in adata.obs[sample_col].unique()
                                           if s.split("_")[0] in liver_biopsy_ids], size=105, title="liver biopsies")

## 4. Annotate

In [ ]:
# --- annotation: leiden_scvi cluster -> label ---
cell_state_labels = {
    "0": "CD8 T naive", "1": "CD4 T regulatory", "2": "CD8 Tem", "3": "CD4 Tem", "4": "CD4 T naive",
    "5": "B cell", "6": "Cytotoxic NK cell", "7": "CD8 Tem_ENTPD1high", "8": "Plasma cell",
    "9": "B cell dividing", "10": "CD4 Tem", "11": "NK cell", "12": "Plasma cell",
}

coarse_labels = {"CD8 Tem_ENTPD1high": "CD8 Tem"}            # cell_state -> cell_state_v2

cell_state_order = [
    "CD4 T naive", "CD4 Tem", "CD4 T regulatory", "CD8 T naive", "CD8 Tem", "CD8 Tem_ENTPD1high",
    "Cytotoxic NK cell", "NK cell", "B cell", "Plasma cell", "B cell dividing",
]

cell_state_colors = {
    "NK cell": "#6161d6", "Cytotoxic NK cell": "#87CEFA", "CD4 T naive": "#917366", "CD4 Tem": "#aa4315",
    "CD4 T regulatory": "#3f1301", "CD8 Tem": "#ef9708", "CD8 Tem_ENTPD1high": "#ea9678", "CD8 T naive": "#fc5e06",
    "B cell": "#068478", "Plasma cell": "#0fcfc0", "B cell dividing": "#97c4bf",
}

cell_state_v2_order = [s for s in cell_state_order if s not in coarse_labels]
cell_state_v2_colors = {s: cell_state_colors[s] for s in cell_state_v2_order}


unmapped = set(adata.obs["leiden_scvi"].astype(str)) - set(cell_state_labels)
assert not unmapped, f"leiden clusters without a label: {sorted(unmapped, key=int)} (cluster IDs changed?)"
adata.obs["cell_state"] = pd.Categorical(adata.obs["leiden_scvi"].astype(str).map(cell_state_labels),
                                         categories=cell_state_order)
adata.obs["cell_state_v2"] = pd.Categorical(adata.obs["cell_state"].astype(str).replace(coarse_labels),
                                            categories=cell_state_v2_order)
check_object(adata, "annotated")

for name in gene_panels:
    sc.pl.umap(adata, color=present(gene_panels[name], adata), ncols=3, wspace=0.3, size=60,
               color_map="OrRd", palette=cell_state_colors)

## 5. Publication UMAPs and marker dotplots (all samples and lung only)

In [ ]:
is_liver = adata.obs["ID"].isin(liver_biopsy_ids)
sc.set_figure_params(figsize=(8, 8))

sc.pl.umap(adata, color="cell_state", size=150, palette=cell_state_colors, legend_fontsize="x-small",
           legend_fontoutline=True, save="_cell_type.pdf") #Figure 5

sc.pl.umap(adata[~is_liver], color="cell_state", size=150, palette=cell_state_colors, legend_fontsize="x-small",
           legend_fontoutline=True, save="_cell_type_regress_out_liver.pdf") #Supplementary Figure 6

sc.pl.umap(adata[is_liver], color="cell_state", size=150, palette=cell_state_colors,
           save="_cell_state_only_liverBiopsy.pdf") #Supplementary Figure 6

sc.set_figure_params(figsize=(4, 4))

dotplot_state_order = [                                         # row order of the publication dotplots
    "Plasma cell", "B cell", "B cell dividing", "NK cell", "Cytotoxic NK cell",
    "CD4 T naive", "CD4 Tem", "CD4 T regulatory", "CD8 T naive", "CD8 Tem", "CD8 Tem_ENTPD1high",
]

plt.rcParams.update({"font.size": 16})
for subset, suffix in ((adata, ""), (adata[~is_liver].copy(), "_lung_only")):
    italic_dotplot(subset, markers_1, "cell_state", (10, 3.2), categories_order=dotplot_state_order,
                   save_path=f"{figures_output_dir}/dotplot_markers_1{suffix}.pdf") #Figure 5
    italic_dotplot(subset, markers_2, "cell_state", (8, 3.2), categories_order=dotplot_state_order,
                   save_path=f"{figures_output_dir}/dotplot_markers_2{suffix}.pdf")
plt.rcParams.update(plt.rcParamsDefault)
sc.set_figure_params(dpi=80, figsize=(4, 4))

## 6. Composition per sample and treatment stage

`masked_samples` are drawn as empty bars in the per-sample % plot (their counts are
printed below); the count barplot shows all samples.

In [ ]:
sample_order = [
    "TN004_AN", "TN002_AN", "TN001_AN", "TN008_AN", "TN006_AN", "TN009_AN", "TN003_AN", "TN005_AN", "TN007_AN",
    "TN008", "TN004", "TN006", "TN002", "TN005", "TN001", "TN007", "TN009", "TN003",
    "P002", "P008", "P003", "O001", "E001", "P013", "P006", "P012", "P005", "P011", "P001", "P009",
    "D001", "D003", "D002",
    "R012", "R006", "R003", "R001", "R010", "R007", "R002", "R004", "R009", "R005", "R008", "R011",
]

masked_samples = ["TN008", "P002", "P008", "P003", "R012", "R006", "R003"]   # shown empty in the per-sample % plot

cell_counts = adata.obs[sample_col].value_counts().reindex([s for s in sample_order if s in set(adata.obs[sample_col])])
print("cells in masked samples:", cell_counts.reindex(masked_samples).to_dict())

fig, (ax_top, ax_bot) = plt.subplots(2, 1, sharex=True, figsize=(10, 5.1), gridspec_kw={"height_ratios": [1, 2], "hspace": 0.05})
for ax in (ax_top, ax_bot):
    ax.bar(range(len(cell_counts)), cell_counts.values, color="black", width=0.8)
    ax.grid(False)
ax_top.set_ylim(count_axis_break, cell_counts.max() * 1.1)
ax_bot.set_ylim(0, count_axis_break)
ax_top.spines["bottom"].set_visible(False); ax_bot.spines["top"].set_visible(False)
ax_top.tick_params(bottom=False)
d = 0.015
for ax, y in ((ax_top, 0), (ax_bot, 1)):
    for x in (0, 1):
        ax.plot((x - d, x + d), (y - d, y + d), transform=ax.transAxes, color="k", clip_on=False)
ax_bot.set_xticks(range(len(cell_counts)), cell_counts.index, rotation=90)
ax_bot.set_ylabel("Lymphoid cells")
fig.savefig(f"{figures_output_dir}/barplot_{sample_col}_lymphoid_cell_counts_ybreak.pdf", bbox_inches="tight", dpi=300)
plt.show()

proportion_barplot(adata, "cell_state", sample_col, cell_state_colors, order=sample_order,
                   category_order=cell_state_order, masked=masked_samples,
                   save_name=f"barplot_{sample_col}_proportions_cell_state.pdf") #Supplementary Figure 6

proportion_barplot(adata, "cell_state", recist_col, cell_state_colors, category_order=cell_state_order,
                   horizontal=True, figsize=(4, 2), save_name=f"barplot_{recist_col}_proportions_cell_state.pdf") 

proportion_barplot(adata, "cell_state_v2", recist_col, cell_state_v2_colors, category_order=cell_state_v2_order,
                   horizontal=True, figsize=(4, 2), save_name=f"barplot_{recist_col}_proportions_cell_state_v2.pdf")

## 7. Final-label markers

In [ ]:
for label in ("cell_state", "cell_state_v2"):
    rank_markers(adata, label, n_genes=10)
    sc.pl.dotplot(adata, present(marker_genes, adata), groupby=label, use_raw=False,
                  standard_scale="var", cmap="Blues", figsize=(30, 7), dendrogram=True)

## 8. Save

In [ ]:
for key in [k for k in adata.uns if k.endswith("_filtered")]:
    del adata.uns[key]
adata.obs[["leiden_scvi", "cell_state", "cell_state_v2"]].to_csv(annotation_csv)
adata.X = adata.layers[counts_layer].copy()
adata.uns.pop("log1p", None)
adata.write(output_path, compression="gzip")
check_object(adata, "saved")